# Data Pipeline for Senate LDA Input Filing Data
## aka, What Lobbying firms are people working with and on what issues?
---



## Input Requirements
- To run this pipeline you will need to first get a csv from lobbyfinder.py
- This pipeline is designed to use all of the inbuild column names from lobbyfinder.py so I don't think it will work with anything else

## To Use
- Make sure all requirements are installed, as I installed the ones I needed as I went
- import your .csv into this folder and change the pdf.readcsv to your .csv name

## Output
- This will leave you with 4 graphics in the Viz folder all built with ploty so that you can interact with them in the notebook to check values and dates. 

### Libraries Used
- pandas, plotly, kaledio, nbformat

### DataFrames in this Pipeline

---
### Last Update: 04/26/26
---

## Step 1: Dependencies and Importing Data

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import sys
!{sys.executable} -m pip install kaleido

import plotly.io as pio
pio.kaleido.scope.mathjax = None


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\craig\AppData\Local\Temp\ipykernel_29228\2036799430.py:9: DeprecationWarning: 
Use of plotly.io.kaleido.scope.mathjax is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.mathjax instead.

  pio.kaleido.scope.mathjax = None


In [3]:
df = pd.read_csv('lobbying_filings_Primes_cleaned - lobbying_filings_Primes.csv')
df.head()

,source,query_company,client_name,registrant_name,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_id,client_id,income,expenses,lobbying_issues,lobbying_bills,lobbying_gov_entities,url
0,Senate LDA,boeing,AVIATION PARTNERS BOEING,DENNY MILLER ASSOCIATES,client,6d2f058c-6a4b-40d2-ae23-10bfd88e20a5,2007,NaN,Year-End Report,2008-02-05T16:13:56.243000-05:00,25328,133127,60000.0,NaN,Taxation/Internal Revenue Code; Transportation...,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/6...
1,Senate LDA,boeing,AVIATION PARTNERS BOEING,DENNY MILLER ASSOCIATES,client,1debd153-0f61-4c3c-babf-316cdea5d06f,2008,NaN,4th Quarter - Report,2009-01-15T12:20:12-05:00,25328,133127,30000.0,NaN,Aviation/Airlines/Airports; Defense; Budget/Ap...,NaN,"Defense, Dept of (DOD); HOUSE OF REPRESENTATIV...",https://lda.senate.gov/filings/public/filing/1...
2,Senate LDA,boeing,AVIATION PARTNERS BOEING,DENNY MILLER ASSOCIATES,client,819f98b2-fcc5-4fff-b880-9036932ec102,2009,NaN,4th Quarter - Report,2010-01-12T16:20:52.677000-05:00,25328,133127,30000.0,NaN,Taxation/Internal Revenue Code; Aviation/Airli...,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/8...
3,Senate LDA,boeing,AVIATION PARTNERS BOEING,DENNY MILLER ASSOCIATES,client,7e151847-d21e-4487-a4f8-970da4108e78,2010,NaN,4th Quarter - Report,2011-01-18T15:51:09.803000-05:00,25328,133127,20000.0,NaN,Defense,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/7...
4,Senate LDA,boeing,AVIATION PARTNERS BOEING,DENNY MILLER ASSOCIATES,client,e98d34b6-7e83-4a6c-8425-7614494b9601,2011,NaN,4th Quarter - Report,2012-01-19T17:05:47-05:00,25328,133127,20000.0,NaN,Defense,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/e...


## Step 2 - Data Cleaning

In [5]:
# Filter for [role] = client

df_clean = df[df['role'] == 'client']


In [6]:
# Finding each Lobbying firm by Filing year

df_clean.groupby(['registrant_name', 'filing_year'])['income'].sum().reset_index().head(10).style.format({'income': '${:,.0f}'})


,registrant_name,filing_year,income
0,262 STRATEGY LLC,2025,$0
1,"535 GROUP, LLC",2023,"$30,000"
2,"535 GROUP, LLC",2024,"$30,000"
3,"535 GROUP, LLC",2025,"$30,000"
4,"535 GROUP, LLC",2026,"$30,000"
5,AB MANAGEMENT ASSOCIATES,2003,$0
6,AB MANAGEMENT ASSOCIATES,2004,$0
7,AB MANAGEMENT ASSOCIATES,2005,$0
8,AB MANAGEMENT ASSOCIATES,2006,$0
9,AB MANAGEMENT ASSOCIATES,2007,"$11,000"


In [7]:
# Total Income by firm per client

df_clean.groupby(['registrant_name', 'query_company'])['income'].sum().reset_index().head(10).style.format({'income': '${:,.0f}'})


,registrant_name,query_company,income
0,262 STRATEGY LLC,thales,$0
1,"535 GROUP, LLC",Leidos,"$120,000"
2,AB MANAGEMENT ASSOCIATES,Lockheed Martin,"$61,000"
3,ACG ADVOCACY,General Dynamics,"$530,000"
4,ACG ADVOCACY,Lockheed Martin,"$50,000"
5,ACG ADVOCACY,honeywell,"$150,000"
6,AKERMAN LLP,honeywell,"$110,000"
7,AKIN GUMP STRAUSS HAUER & FELD,RTX,"$1,040,000"
8,AKIN GUMP STRAUSS HAUER & FELD,boeing,"$1,820,000"
9,AKIN GUMP STRAUSS HAUER & FELD,honeywell,"$2,150,000"


## Step 3 - Vizualization

In [8]:
df_sankey = df_clean.groupby(['query_company', 'registrant_name'])['income'].sum().reset_index()
df_sankey = df_sankey[df_sankey['income'] > 0].dropna(subset=['income'])
df_sankey.head()


,query_company,registrant_name,income
0,General Dynamics,ACG ADVOCACY,530000.0
1,General Dynamics,ALIGNMENT GOVERNMENT STRATEGIES,400000.0
2,General Dynamics,AMERICAN DEFENSE INTERNATIONAL,720000.0
4,General Dynamics,"AVOQ, LLC",400000.0
5,General Dynamics,BAKER DONELSON BEARMAN CALDWELL & BERKOWITZ,225000.0


In [9]:
clients = df_sankey['query_company'].unique().tolist()
firms = df_sankey['registrant_name'].unique().tolist()
nodes = clients + firms


In [10]:
source = [nodes.index(c) for c in df_sankey['query_company']]
target = [nodes.index(f) for f in df_sankey['registrant_name']]
value = df_sankey['income'].tolist()


In [11]:
# Limit to top 10 clients and top 15 firms by income for readability
top_clients = df_sankey.groupby('query_company')['income'].sum().nlargest(10).index
top_firms = df_sankey.groupby('registrant_name')['income'].sum().nlargest(15).index

df_sankey_top = df_sankey[
    df_sankey['query_company'].isin(top_clients) &
    df_sankey['registrant_name'].isin(top_firms)
]

# Clients on left (source), firms on right (target) — both sorted largest to top
clients = (df_sankey_top.groupby('query_company')['income'].sum()
           .sort_values(ascending=False).index.tolist())
firms = (df_sankey_top.groupby('registrant_name')['income'].sum()
         .sort_values(ascending=False).index.tolist())
nodes = clients + firms

source = [nodes.index(c) for c in df_sankey_top['query_company']]
target = [nodes.index(f) for f in df_sankey_top['registrant_name']]
value = df_sankey_top['income'].tolist()

fig = go.Figure(go.Sankey(
    node=dict(label=nodes, pad=25, thickness=20, color='steelblue'),
    link=dict(source=source, target=target, value=value, color='rgba(100,149,237,0.3)')
))
fig.update_layout(
    title='Top Lobbying Firm Income by Client',
    height=700,
    font=dict(size=11),
    hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey')
)
fig.show()